# Extraction with raw and log costs (correctly built τ)

Builds two τ structures, *both* averaged over phis with `phi(j)=u` but using different transforms:
- $\tau[i,j,v,u] = \mathbb{E}_{\varphi'}[\hat P[\psi(i)=v \mid \varphi']]$  (raw)
- $\tau_{\log}[i,j,v,u] = \mathbb{E}_{\varphi'}[\log \hat P[\psi(i)=v \mid \varphi']]$  (log-then-average; **NOT** $\log\tau$, since by Jensen $\log E[X] \neq E[\log X]$).

Witness coordinates are picked separately for each space (raw spread on $\tau$, log spread on $\tau_{\log}$). The extractor then runs all four method combinations:
- max-spread witness  +  raw cost
- max-min-spread witness  +  raw cost
- max-spread witness (log)  +  log cost
- max-min-spread witness (log)  +  log cost

Summary reports per-method top-1 / top-N / mean rank, plus a **union** row (any method got it).

In [1]:
import sys, random, itertools
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn.functional as F
import numpy as np
from scipy.optimize import linear_sum_assignment

import config
from model import TinyTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N = config.N
EPS = 1e-12
print(f'N = {N}, device = {DEVICE}')

N = 4, device = cuda


In [2]:
CHECKPOINT = PROJECT_ROOT / 'checkpoints' / 'model_2500.pt'

def load_model(path):
    m = TinyTransformer().to(DEVICE)
    m.load_state_dict(torch.load(path, map_location=DEVICE))
    m.eval()
    return m

model = load_model(CHECKPOINT)
print(f'Loaded {CHECKPOINT.name}')

Loaded model_2500.pt


/home/akash10/miniconda3/envs/aug-spm/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/tmp/ipykernel_691894/2073868983.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for

In [3]:
@torch.no_grad()
def sample_psi_unconstrained(model, phis):
    model.eval()
    if phis.dim() == 1:
        phis = phis.unsqueeze(0)
    B = phis.shape[0]
    psi_start = config.PSI.start
    psi_buf = torch.zeros((B, N), dtype=torch.long, device=DEVICE)
    seq = torch.cat([phis.to(DEVICE), psi_buf], dim=1)
    for j in range(N):
        logits = model(seq)
        step_logits = logits[:, psi_start - 1 + j, :]
        probs = F.softmax(step_logits, dim=-1)
        sampled = torch.multinomial(probs, num_samples=1).squeeze(-1)
        seq[:, psi_start + j] = sampled
    return seq[:, psi_start:psi_start + N]

def is_perm_rows(samples, n):
    sorted_, _ = torch.sort(samples, dim=1)
    expected = torch.arange(n, device=samples.device).unsqueeze(0).expand_as(sorted_)
    return (sorted_ == expected).all(dim=1)

@torch.no_grad()
def sample_psi_rejection(model, phis, max_attempts=50):
    """Whole-sequence rejection: resample any row whose psi isn't a permutation."""
    model.eval()
    if phis.dim() == 1:
        phis = phis.unsqueeze(0)
    B = phis.shape[0]
    phis = phis.to(DEVICE)
    out     = torch.zeros((B, N), dtype=torch.long, device=DEVICE)
    pending = torch.ones(B,        dtype=torch.bool, device=DEVICE)
    for _ in range(max_attempts):
        idx = torch.nonzero(pending, as_tuple=False).squeeze(1)
        if idx.numel() == 0:
            break
        psis_try = sample_psi_unconstrained(model, phis[idx])
        valid = is_perm_rows(psis_try, N)
        out[idx[valid]] = psis_try[valid]
        pending[idx[valid]] = False
    if pending.any():
        raise RuntimeError(
            f'{int(pending.sum().item())} rows still invalid after {max_attempts} attempts'
        )
    return out

def marginal_matrix(samples, n=N):
    return F.one_hot(samples.long(), n).float().mean(dim=0)

def random_phis_with_constraint(B, j, u, n=N, seed=None):
    g = torch.Generator()
    if seed is not None: g.manual_seed(seed)
    other_positions = torch.tensor([i for i in range(n) if i != j], dtype=torch.long)
    other_values    = torch.tensor([v for v in range(n) if v != u], dtype=torch.long)
    keys = torch.rand((B, n - 1), generator=g)
    perms = keys.argsort(dim=1)
    perm_values = other_values[perms]
    out = torch.zeros((B, n), dtype=torch.long)
    out[:, j] = u
    out[:, other_positions] = perm_values
    return out

In [4]:
def best_assignment(C):
    C_np = C.cpu().numpy() if torch.is_tensor(C) else C
    row_ind, col_ind = linear_sum_assignment(C_np)
    return torch.tensor(col_ind, dtype=torch.long), float(C_np[row_ind, col_ind].sum())

def top_k_assignments(C, k=None):
    """Top-k lowest-cost assignments via brute-force enumeration of S_n."""
    n = C.shape[0]
    C_np = C.cpu().numpy() if torch.is_tensor(C) else C
    rows = np.arange(n)
    scored = [
        (C_np[rows, list(perm)].sum(), perm)
        for perm in itertools.permutations(range(n))
    ]
    scored.sort(key=lambda x: x[0])
    if k is not None:
        scored = scored[:k]
    return [(torch.tensor(p, dtype=torch.long), float(c)) for c, p in scored]

## Build τ and τ_log

For each `(j, u)` cell:
1. Sample `K1` phis with `phi(j)=u`.
2. For each phi, sample `K2` psis (via whole-seq rejection so every psi is a valid permutation).
3. Compute per-phi marginal `M_k[i, v] = (1/K2) Σ 1[ψ(i)=v]`.
4. Average `M_k` and `log(M_k)` over the K1 phis.

Sampling cost per cell: `K1 × K2` model forwards (batched into one big call). Watch GPU memory — K1·K2 ≈ 100k is comfortable, much more may need chunking.

In [ ]:
tau     = {}      # E_phi[ P[psi(i)=v | phi] ]
tau_log = {}      # E_phi[ log P[psi(i)=v | phi] ]

seed = 43
torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
np.random.seed(seed); random.seed(seed)

K1 = 2**15          # phis per (j, u)
K2 = 2**15          # psis per phi (large to keep log-bias small)

# Memory bound: one forward pass batch is at most CHUNK rows.
# Chunk over phis first, then over psis within each phi-chunk.
CHUNK = 2**14     # rows per model.forward call — tune for your GPU

for j in range(N):
    for u in range(N):
        phis = random_phis_with_constraint(K1, j=j, u=u)              # (K1, N)

        sum_p     = torch.zeros((N, N), device=DEVICE)
        sum_log_p = torch.zeros((N, N), device=DEVICE)

        # PHI_CHUNK = how many phis to process at once. Each phi needs K2 psis,
        # so the per-call batch is PHI_CHUNK * PSI_SUBCHUNK ≤ CHUNK.
        # Pick the largest PHI_CHUNK such that PHI_CHUNK * K2 fits.
        # If K2 itself > CHUNK, fall back to processing one phi at a time and
        # splitting K2 across multiple sub-chunks.
        if K2 <= CHUNK:
            phi_chunk = max(1, CHUNK // K2)
            psi_subchunk = K2
        else:
            phi_chunk = 1
            psi_subchunk = CHUNK

        for phi_start in range(0, K1, phi_chunk):
            P = min(phi_chunk, K1 - phi_start)
            this_phis = phis[phi_start:phi_start + P]                  # (P, N)

            # Counts of (psi(i)=v) per phi-in-chunk, accumulated over psi sub-chunks.
            counts = torch.zeros((P, N, N), device=DEVICE)
            for psi_start in range(0, K2, psi_subchunk):
                S = min(psi_subchunk, K2 - psi_start)
                phi_rep = this_phis.repeat_interleave(S, dim=0)        # (P*S, N)
                psis    = sample_psi_rejection(model, phi_rep)         # (P*S, N) on DEVICE
                psis    = psis.view(P, S, N)
                counts += F.one_hot(psis.long(), N).float().sum(dim=1) # (P, N, N)

            M_per_phi = counts / K2                                    # (P, N, N) per-phi marginals
            sum_p     += M_per_phi.sum(dim=0)
            sum_log_p += torch.log(M_per_phi.clamp_min(EPS)).sum(dim=0)

        avg_p     = sum_p / K1
        avg_log_p = sum_log_p / K1

        for i in range(N):
            for v in range(N):
                tau[(i, j, v, u)]     = avg_p[i, v]
                tau_log[(i, j, v, u)] = avg_log_p[i, v]
        print(f'  cell (j={j}, u={u}) done')

  cell (j=0, u=0) done
  cell (j=0, u=1) done
  cell (j=0, u=2) done
  cell (j=0, u=3) done
  cell (j=1, u=0) done
  cell (j=1, u=1) done
  cell (j=1, u=2) done
  cell (j=1, u=3) done
  cell (j=2, u=0) done
  cell (j=2, u=1) done
  cell (j=2, u=2) done
  cell (j=2, u=3) done
  cell (j=3, u=0) done
  cell (j=3, u=1) done
  cell (j=3, u=2) done
  cell (j=3, u=3) done


## Pick witnesses (raw-space spreads on τ, log-space spreads on τ_log)

Two scoring rules per space:
- **max spread** = max_u value − min_u value (range)
- **max-min spread** = top-1 − top-2 (margin to runner-up)

In [6]:
def pick_witnesses(tau_dict):
    """Return (opts_max_spread, opts_max_min_spread): each {j: (i_j, v_j)}."""
    opts_ms, opts_mm = {}, {}
    for j in range(N):
        scores_ms = torch.zeros((N, N))
        scores_mm = torch.zeros((N, N))
        for i in range(N):
            for v in range(N):
                u_func = torch.tensor([tau_dict[(i, j, v, u)].item()
                                       if torch.is_tensor(tau_dict[(i, j, v, u)])
                                       else tau_dict[(i, j, v, u)]
                                       for u in range(N)])
                scores_ms[i, v] = u_func.max() - u_func.min()
                sorted_, _ = torch.sort(u_func, descending=True)
                scores_mm[i, v] = sorted_[0] - sorted_[1]      # top-1 minus top-2
        i_ms, v_ms = (scores_ms.argmax() // N).item(), (scores_ms.argmax() % N).item()
        i_mm, v_mm = (scores_mm.argmax() // N).item(), (scores_mm.argmax() % N).item()
        opts_ms[j], opts_mm[j] = (i_ms, v_ms), (i_mm, v_mm)
    return opts_ms, opts_mm

opts_max_spread,     opts_max_min_spread     = pick_witnesses(tau)
opts_max_spread_log, opts_max_min_spread_log = pick_witnesses(tau_log)

print('raw spreads — opts_max_spread:    ', opts_max_spread)
print('raw spreads — opts_max_min_spread:', opts_max_min_spread)
print('log spreads — opts_max_spread:    ', opts_max_spread_log)
print('log spreads — opts_max_min_spread:', opts_max_min_spread_log)

raw spreads — opts_max_spread:     {0: (2, 2), 1: (0, 1), 2: (1, 3), 3: (0, 2)}
raw spreads — opts_max_min_spread: {0: (0, 3), 1: (2, 3), 2: (1, 3), 3: (1, 3)}
log spreads — opts_max_spread:     {0: (3, 2), 1: (0, 3), 2: (1, 1), 3: (0, 1)}
log spreads — opts_max_min_spread: {0: (2, 2), 1: (0, 1), 2: (1, 0), 3: (0, 2)}


## Run extraction over all permutations

Four methods. Per-iteration prints the top-N for each method and the rank of truth in each. Summary at the end shows top-1, top-N, mean rank for each method, plus a **union** row (any method got truth in top-1 / top-N).

In [7]:
B = K2                                              # psi samples per test phi (independent of K1, K2)
B_CHUNK = CHUNK                                          # forward-pass batch size — tune for GPU
permutations = list(itertools.permutations(range(N)))

seed = 1
torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
np.random.seed(seed); random.seed(seed)

# (witness_dict, tau_source, cost_transform_label)
METHODS = {
    'max-min raw': (opts_max_min_spread,     tau,     'raw'),
    'max     raw': (opts_max_spread,         tau,     'raw'),
    'max-min log': (opts_max_min_spread_log, tau_log, 'log'),
    'max     log': (opts_max_spread_log,     tau_log, 'log'),
}

ranks = {m: [] for m in METHODS}
top1  = {m: 0  for m in METHODS}
topN  = {m: 0  for m in METHODS}
union_top1 = 0
union_topN = 0

for permutation in permutations:
    # Sample B psis for this fixed phi, in chunks of B_CHUNK.
    base = torch.tensor(permutation, device=DEVICE).long()             # (N,)
    counts = torch.zeros((N, N), device=DEVICE)
    for start in range(0, B, B_CHUNK):
        S = min(B_CHUNK, B - start)
        perm_rep = base.unsqueeze(0).expand(S, -1).contiguous()        # (S, N)
        psis = sample_psi_rejection(model, perm_rep)                   # (S, N) on DEVICE
        counts += F.one_hot(psis.long(), N).float().sum(dim=0)         # (N, N)
    mm     = counts / B                                                 # (N, N)
    log_mm = torch.log(mm.clamp_min(EPS))

    truth_tup = tuple(permutation)
    print(f'truth = {truth_tup}')
    iter_in_top1 = iter_in_topN = False

    for name, (opts_dict, tau_dict, kind) in METHODS.items():
        C = torch.zeros((N, N))
        for j in range(N):
            i_j, v_j = opts_dict[j]
            if kind == 'raw':
                p_j = mm[i_j, v_j]
                for u in range(N):
                    C[j, u] = torch.abs(tau_dict[(i_j, j, v_j, u)] - p_j)
            else:                                                       # 'log'
                lp_j = log_mm[i_j, v_j]
                for u in range(N):
                    C[j, u] = torch.abs(tau_dict[(i_j, j, v_j, u)] - lp_j)

        full   = top_k_assignments(C, k=None)
        top_N  = full[:N]
        rank   = next(i + 1 for i, (p, _) in enumerate(full) if tuple(p.tolist()) == truth_tup)
        in_top = rank <= N
        ranks[name].append(rank)
        if rank == 1:
            top1[name] += 1
            iter_in_top1 = True
        if in_top:
            topN[name] += 1
            iter_in_topN = True
        print(f'  {name:<14}  best={full[0][0].tolist()} (cost {full[0][1]:.4f})  '
              f'rank={rank}/{len(permutations)}  in-top-{N}={in_top}')
        print(f'    top-{N}: {[(p.tolist(), round(c, 4)) for p, c in top_N]}')
    if iter_in_top1: union_top1 += 1
    if iter_in_topN: union_topN += 1
    print()

truth = (0, 1, 2, 3)
  max-min raw     best=[1, 2, 3, 0] (cost 0.1169)  rank=8/24  in-top-4=False
    top-4: [([1, 2, 3, 0], 0.1169), ([3, 2, 1, 0], 0.1752), ([3, 1, 2, 0], 0.2162), ([1, 0, 2, 3], 0.2243)]
  max     raw     best=[0, 2, 1, 3] (cost 0.3729)  rank=9/24  in-top-4=False
    top-4: [([0, 2, 1, 3], 0.3729), ([1, 2, 3, 0], 0.3871), ([1, 0, 3, 2], 0.3966), ([1, 2, 0, 3], 0.4288)]
  max-min log     best=[3, 2, 1, 0] (cost 22.4964)  rank=22/24  in-top-4=False
    top-4: [([3, 2, 1, 0], 22.4964), ([1, 2, 3, 0], 24.4225), ([2, 1, 3, 0], 27.5097), ([2, 3, 1, 0], 33.5804)]
  max     log     best=[1, 2, 3, 0] (cost 16.9365)  rank=11/24  in-top-4=False
    top-4: [([1, 2, 3, 0], 16.9365), ([2, 1, 3, 0], 21.3396), ([2, 0, 1, 3], 22.0217), ([0, 2, 1, 3], 23.9549)]

truth = (0, 1, 3, 2)
  max-min raw     best=[1, 3, 0, 2] (cost 0.3834)  rank=7/24  in-top-4=False
    top-4: [([1, 3, 0, 2], 0.3834), ([0, 3, 1, 2], 0.5871), ([1, 3, 2, 0], 0.7112), ([3, 1, 0, 2], 0.7151)]
  max     raw     be

In [8]:
T = len(permutations)
print('=' * 80)
print(f'Summary over all {T} permutations of S_{N}')
print(f'Random baselines: top-1 ≈ {100/T:.1f}%, top-{N} ≈ {100*N/T:.1f}%, mean rank ≈ {(T+1)/2:.1f}')
print('-' * 80)
print(f'{"method":<20} {"top-1":>10} {"top-N":>10} {"mean rank":>14}')
for m in METHODS:
    print(f'{m:<20} {top1[m]}/{T:<6} {topN[m]}/{T:<6} {sum(ranks[m])/T:>12.2f}')
print('-' * 80)
print(f'{"union (any method)":<20} {union_top1}/{T:<6} {union_topN}/{T:<6}')

Summary over all 24 permutations of S_4
Random baselines: top-1 ≈ 4.2%, top-4 ≈ 16.7%, mean rank ≈ 12.5
--------------------------------------------------------------------------------
method                    top-1      top-N      mean rank
max-min raw          2/24     9/24             7.17
max     raw          7/24     11/24             6.54
max-min log          2/24     10/24             9.00
max     log          5/24     12/24             6.46
--------------------------------------------------------------------------------
union (any method)   13/24     20/24    


In [9]:
================================================================================
Summary over all 24 permutations of S_4
Random baselines: top-1 ≈ 4.2%, top-4 ≈ 16.7%, mean rank ≈ 12.5
--------------------------------------------------------------------------------
method                    top-1      top-N      mean rank
max-min raw          3/24     12/24             7.46
max     raw          3/24     12/24             7.46
max-min log          4/24     8/24             8.46
max     log          2/24     10/24             7.25
--------------------------------------------------------------------------------
union (any method)   7/24     19/24    

SyntaxError: invalid character '≈' (U+2248) (1148806139.py, line 3)

In [ ]:
================================================================================
Summary over all 24 permutations of S_4
Random baselines: top-1 ≈ 4.2%, top-4 ≈ 16.7%, mean rank ≈ 12.5
--------------------------------------------------------------------------------
method                    top-1      top-N      mean rank
max-min raw          3/24     11/24             7.67
max     raw          3/24     11/24             7.67
max-min log          2/24     8/24             9.38
max     log          2/24     6/24            10.42
--------------------------------------------------------------------------------
union (any method)   5/24     16/24    

In [ ]:
================================================================================
Summary over all 24 permutations of S_4
Random baselines: top-1 ≈ 4.2%, top-4 ≈ 16.7%, mean rank ≈ 12.5
--------------------------------------------------------------------------------
method                    top-1      top-N      mean rank
max-min raw          4/24     9/24             8.88
max     raw          4/24     12/24             6.92
max-min log          2/24     14/24             7.96
max     log          2/24     7/24             8.58
--------------------------------------------------------------------------------
union (any method)   7/24     21/24    

SyntaxError: invalid character '≈' (U+2248) (582403741.py, line 3)